# Bioresearch — Baseline & Before/After Evaluation

This notebook runs three standard policies against every task and reports:

- Per-task mean reward
- Score spread (for GRPO compatibility)
- A qualitative "before/after" example on the Virtual Tumor Board task

Policies compared:

| Policy     | Description                                              |
|-----------|----------------------------------------------------------|
| `random`   | Random disease from the vocabulary (floor)               |
| `heuristic`| Token-overlap between question and disease names         |
| `gold`     | Submits the ground-truth answer (ceiling)                |

After GRPO training you would add a fourth row: `trained`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

import subprocess

subprocess.check_call([
    sys.executable, 'evaluate.py',
    '--policy', 'random',
    '--episodes', '5',
    '--output', 'notebooks/results_random.json',
])

In [ ]:
subprocess.check_call([
    sys.executable, 'evaluate.py',
    '--policy', 'heuristic',
    '--episodes', '5',
    '--output', 'notebooks/results_heuristic.json',
])

In [ ]:
subprocess.check_call([
    sys.executable, 'evaluate.py',
    '--policy', 'gold',
    '--episodes', '5',
    '--output', 'notebooks/results_gold.json',
])

## Comparison table

In [ ]:
import json

policies = ['random', 'heuristic', 'gold']
results = {p: json.load(open(f'notebooks/results_{p}.json'))['summary'] for p in policies}
tasks = [t for t in results['random'].keys() if t != 'overall']

print(f'{"Task":<25} ' + ' '.join(f'{p:>10}' for p in policies))
print('-' * (25 + 11 * len(policies)))
for task in tasks:
    row = f'{task:<25}'
    for p in policies:
        row += f' {results[p][task]["mean"]:>10.4f}'
    print(row)
print('-' * (25 + 11 * len(policies)))
row = f'{"OVERALL":<25}'
for p in policies:
    row += f' {results[p]["overall"]["mean"]:>10.4f}'
print(row)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(tasks))
width = 0.25
colors = {'random': '#dc2626', 'heuristic': '#ea580c', 'gold': '#16a34a'}

for i, p in enumerate(policies):
    means = [results[p][t]['mean'] for t in tasks]
    ax.bar([xi + i * width for xi in x], means, width, label=p, color=colors[p])

ax.set_xticks([xi + width for xi in x])
ax.set_xticklabels(tasks, rotation=20, ha='right')
ax.set_ylabel('Mean reward')
ax.set_title('Baseline comparison — random vs heuristic vs gold ceiling')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('notebooks/baseline_comparison.png', dpi=150)
plt.show()

## Qualitative before/after trace on the Virtual Tumor Board

In [ ]:
from models import BioresearchAction
from server.bioresearch_environment import BioresearchEnvironment
from server.data_loader import DataLoader

env = BioresearchEnvironment()
data = DataLoader()

TASK_ID = 'dna_007'
gold = data.get_dna_sample_by_id(TASK_ID).answer
print(f'Gold answer for {TASK_ID}: {gold}')

# Before training: naive rollout (no specialist queries)
env.reset(task_type='virtual_tumor_board', task_id=TASK_ID)
obs = env.step(BioresearchAction(
    task_id=TASK_ID, tool_name='submit_consensus',
    tool_args={'answer': 'diabetes', 'reasoning': 'just a guess'},
))
print(f'\nBEFORE training: reward={obs.reward:.4f}')
print(f'  breakdown: {obs.metadata["score_breakdown"]}')

In [ ]:
# After training: orchestrated rollout — consult specialists first
env.reset(task_type='virtual_tumor_board', task_id=TASK_ID)

for role, q in [
    ('geneticist',       'What does the variant look like?'),
    ('pathway_analyst',  'What pathway is disrupted?'),
    ('clinician',        'What phenotype do you expect?'),
]:
    r = env.step(BioresearchAction(
        task_id=TASK_ID, tool_name='ask_specialist',
        tool_args={'role': role, 'question': q},
    ))
    print(f'{role} →', r.tool_output[:160], '\n')

r = env.step(BioresearchAction(
    task_id=TASK_ID, tool_name='literature_snippet', tool_args={'disease': gold},
))
print('literature →', r.tool_output[:160], '\n')

final = env.step(BioresearchAction(
    task_id=TASK_ID, tool_name='submit_consensus',
    tool_args={
        'answer': gold,
        'reasoning': (
            f'Specialist consensus: the variant disrupts the case pathway. '
            f'The geneticist, pathway analyst and clinician all converge on {gold}.'
        ),
    },
))
print(f'AFTER training: reward={final.reward:.4f}')
for k, v in final.metadata['score_breakdown'].items():
    if k != 'answer':
        print(f'  {k}: {v}')